## **Lo que van a encontrar en este notebook**

Tenemos tres metas:

1. Ver la filtracion de Rips en ejemplos sencillos y calcular sus codigos de barras.

2. Repasar el algoritmo de SW1PerS (Sliding Windows and 1-Persistence Scoring) para cuantificar periodicidad/recurrencia en series de tiempo.

3. Utilizar el algoritmo SW1PerS para ordenar por periodicidad algunas series de tiempo en un conjunto sintetico.

Completar las actividades marcadas con **Hacer** o **Tu Respuesta**. Sugiero leer atentamente el codigo y los comentarios para tener una idea de que esta pasando.


## Parte I: Persistencia Homologica

In [ ]:
import numpy as np

# plotting and visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Numerical
from scipy.spatial import distance

np.random.seed(1223)

n_data = 30
theta = np.random.uniform(0, 2*np.pi, n_data)
data = np.array([np.cos(theta) , np.sin(theta) , np.zeros_like(theta)]).T
data += np.random.normal(0, 0.08, data.shape)

fig = go.Figure(data=[go.Scatter3d(
    x=data.T[0], y=data.T[1], z=data.T[2],
    mode ='markers',
    marker=dict(size = 3 , color = 'grey'))])

fig.update_layout(scene= dict(zaxis = dict(range=[-1, 1])))
fig.show()

### Filtracion de Rips

Dado un espacio metrico $(X,\mathbf{d}_X)$, su filtracion de Rips es:

$$\mathcal{R}(X) = \Big\{\,  R_\alpha(X,\mathbf{d}_X) \Big\}_{ \alpha \geq 0} $$
donde
$$R_\alpha(X, \mathbf{d}_X) = \Big\{\, \{x_0 , \ldots, x_k\} \subseteq X  \;\; : \;\;  \max_{0 \leq i \leq j \leq k} \mathbf{d}_X(x_i, x_j) \leq \alpha\, \Big\} $$

A continuacion vamos a calcular el complejo de Rips de los datos en le circulo para varios valores de $\alpha$

In [ ]:
alpha = 0.1  # cambie este valor!! 0.5, 1.0, 1.75

distMat = distance.squareform(distance.pdist(data))

ii = []; jj = []; kk = []
e_x =[]; e_y =[]; e_z =[]

for i in range(n_data):
    for j in range(i+1,n_data):
        if distMat[i,j] < alpha:
            # add edge (i,j)
            e_x.extend([data[i,0], data[j,0], None])
            e_y.extend([data[i,1], data[j,1], None])
            e_z.extend([data[i,2], data[j,2], None])

            for k in range(j+1,n_data):
                if np.max([distMat[j,k], distMat[i,k]]) < alpha:
                    # add triangle (i,j,k)
                    ii.append(i); jj.append(j); kk.append(k)

vertices = go.Scatter3d(mode = 'markers', name = 'vertices',
                        x = data.T[0], y = data.T[1],  z = data.T[2],
                        marker=dict(size = 3 , color = 'grey'))

edges = go.Scatter3d(mode='lines', name = 'aristas',
                     x=e_x, y=e_y, z=e_z,
                     line=dict(color= 'rgb(70,70,70)', width=1))

triangles = go.Mesh3d(x=data.T[0], y=data.T[1], z=data.T[2],  i = ii, j = jj, k = kk,  color='lightpink', opacity=0.2)

fig = go.Figure(data=[vertices, edges, triangles])
fig.update_traces(hoverinfo="none")
fig.update_layout(scene= dict(
                      xaxis = dict(showspikes=False),
                      yaxis = dict(showspikes=False),
                      zaxis = dict(showspikes=False,range=[-1, 1])))
fig.show()


**Hacer:** Use la celda anterior para visualizar el complejo de Rips de $X$ para varios valores de $\alpha \geq 0 $. Completar los valores faltantes:

1.   $\beta_0(R_\alpha(X, \mathbf{d}_X)) = 5$   cuando  $\alpha = 0.4$

2.   $\beta_1(R_\alpha(X, \mathbf{d}_X)) = 1$   cuando  $\alpha = 1.0$

3.   $\beta_0(R_\alpha(X, \mathbf{d}_X)) = 1$ al mismo tiempo que  $\beta_1(R_\alpha(X, \mathbf{d}_X)) = 0 $
 cuando $\alpha = 1.75$

---

### Calculando codigos de barra via Ripser

U. Bauer: "Ripser is a lean C++ code for the computation of Vietoris–Rips persistence barcodes. It can do just this one thing, but does it extremely well."

Libreria C++  : https://github.com/Ripser/ripser

Libreria de Python library: https://ripser.scikit-tda.org/en/latest

Articulo : https://arxiv.org/pdf/1908.02518.pdf

In [ ]:
!pip install ripser

A continuacion calculamos la persistencia de la filtracion de Rips de los datos usando `Ripser`

In [ ]:
import matplotlib.pyplot as plt

# topological data analysis
from ripser import ripser
from persim import plot_diagrams


def plot_barcodes(diagrams, alpha_max, width = 1.5):
    max_dim = len(diagrams)
    fig, axs = plt.subplots(max_dim)
    fig.suptitle('Barcodes')
    for dim in range(max_dim):
        barcode = np.copy(diagrams[dim])
        ind_inf = np.isinf(barcode.T[1])
        barcode[ind_inf, 1] = alpha_max
        h = 1
        for i in range(len(barcode)):
            x = barcode[i]
            y = [h,h]
            axs[dim].plot(x, y, linestyle= '-', c='#1f77b4', linewidth = width)
            if ind_inf[i]:
                axs[dim].scatter([alpha_max],[h],  s=10, marker='>', c='#1f77b4')
            h += 1
        axs[dim].set_xlim(0, 1.05*alpha_max)
        axs[dim].set_ylim(0,h)
        axs[dim].get_yaxis().set_ticks([]);
        axs[dim].spines['right'].set_color('none')
        axs[dim].spines['top'].set_color('none')
        axs[dim].text(0.3, 1, r'$\mathrm{bcd}^{\mathcal{R}}_{'+str(dim)+'}(X)$', verticalalignment='bottom')


# Persistence Computation
rips_persistence = ripser(data, maxdim=1)

dgms = rips_persistence['dgms']
plot_barcodes(dgms,1.8);

### Diagramas de persistencia

$$ \mathrm{dgm}_j^\mathcal{R}(X) = \Big\{ (a,b) \in \mathbb{R}^2 \;\; : \;\; [a,b) \in \mathrm{bcd}_j^\mathcal{R}(X)  \Big\}$$

In [ ]:
plt.figure(figsize = (3,3))
plot_diagrams(dgms, title='Persistence Diagrams')

---

## Ejemplo: Toro ruidoso

In [ ]:
np.random.seed(2)
n_data = 25000
R = 5
r = 2
data = np.zeros((3, n_data))
s = np.random.rand(n_data)*2*np.pi
t = np.random.rand(n_data)*2*np.pi

data[0] = (R + r*np.cos(s))*np.cos(t)
data[1] = (R + r*np.cos(s))*np.sin(t)
data[2] = r*np.sin(s)
data += 0.1*np.random.randn(*data.shape)
data = data.T

# Plot the data
fig = go.Figure(data=[go.Scatter3d(
    x=data.T[0], y=data.T[1], z=data.T[2],
    mode ='markers',
    marker=dict(size = 1.5 , color = 'grey'))])

fig.update_layout( width=900, height=450)
fig.show()

In [ ]:
## Persistence Computation

# Parameters
max_alpha = 5.35
max_homology_dim = 2

# Run Ripser
rips_persistence = ripser(data,  maxdim=max_homology_dim , thresh = max_alpha , n_perm =200)

# Visualize barcodes
dgms = rips_persistence['dgms']
plot_barcodes(dgms,max_alpha);

In [ ]:
# Plot persistence diagrams
plt.figure(figsize = (3,3))
plot_diagrams(dgms)

**Preguntas**

1.  Sea $T$ el toro y $X$  el conjunto de datos arriba.
Encuentre  $\alpha$ tal que $\beta_j(R_\alpha(X)) = \beta_j(T)$ para $0 \leq j \leq 2$.

2. Que visualizacion fue mas util en esta tarea, los barcodes o los diagramas de persistencia?

**Tu respuesta:**

1. $α = $

2.

---

## Parte II: Ventanas Deslizantes (Sliding Windows)

Dada una serie de tiempo $f: I \subseteq \mathbb{R} \rightarrow  \mathbb{R}$ un parametro de dilacion $\tau > 0$ y una dimension  $d+1 \in \mathbb{N}$, las **ventanas deslizantes** (sliding windows) de $f$ en $t \in I$ es

$$SW_{d,\tau}f(t) = \begin{bmatrix}f(t) \\ f(t + \tau)  \\ \vdots \\ f(t + d\tau)  \end{bmatrix}  \in \mathbb{R}^{d+1}$$

y la **nube de ventanas deslizantes** (sliding window point cloud) es

$$\mathbb{SW}_{d,\tau} f = \{SW_{d,\tau}(t) \;\; : \;\; t\in I\} \subseteq \mathbb{R}^{d+1}$$

In [ ]:
from scipy.interpolate import CubicSpline

def SW_cloud(f, tau, d, n_data):
    # Inputs:
    # f : time series -- array of size (2,N) (x and y values) or (1,N) (only y values)
    # tau: delay -- positive real number
    # d : gives embedding dimension d+1 -- integer
    # n_data : desired number of points in SW point cloud -- intenger
    #
    # Output:
    # SW : sliding window point cloud -- array of size (n_data,  d+1)

    #Step 1: turn f into a cubic spline
    if len(f.shape)==1:
        N = len(f)
        x_vals = np.linspace(0,1,N)
        y_vals = f
    else:
        x_vals = f[0]
        y_vals = f[1]

    f = CubicSpline(x_vals , y_vals)

    #Step 2: create the t values where to evaluate SW_f
    t_vals = np.linspace(np.min(x_vals) , np.max(x_vals) - d*tau, n_data)

    #Step 3: evaluate the sliding window point cloud
    SW = []
    for t in t_vals:
        SW_f_t = f(t + np.arange(0,d+1)*tau)
        SW.append(SW_f_t)

    return np.array(SW)

### Ejemplo:

Una serie de tiempo periodica con un poco de ruido

In [ ]:
# Time Series Example : Noisy sin(t)

period = 2*np.pi   ## <-- sin(t) is periodic with period 2pi
t_vals = np.linspace(0 , 5*period, 1000)
noise_level = 0.1

y_vals  = np.sin(t_vals) + noise_level*np.random.randn(*t_vals.shape)

plt.figure(figsize = (9,2))
plt.plot(t_vals, y_vals);
plt.title('Serie de tiempo');
plt.xlabel('$t$');
plt.ylabel('$f(t)$');


a continuacion calculamos la nube de ventanas deslizantes

In [ ]:
## Compute the Sliding window point cloud

f = np.array([t_vals, y_vals]) ## <---- serie de tiempo

# Parametros para SW
d = 2
tau = period/(d+1)  ## <--- la teoria de SW dice que estos son los mejores parametros

n_data = 5000

# Calcular SW
SW_f = SW_cloud(f,tau, d, n_data )

# Visualizar la nube de ventanas
fig = go.Figure(data=[go.Scatter3d(
    x=SW_f.T[0], y=SW_f.T[1], z=SW_f.T[2],
    mode ='markers',
    marker=dict(size = 1.5 , color = 'grey'))])

fig.update_layout( width=900, height=450)
fig.show()

**Preguntas**
1. Que le pasa a la nube de ventanas deslizantes si se aumenta el ruido de la serie de tiempo? Mire a ver que pasa.


2. Que le pasa a $\mathbb{SW}_{d,\tau} f$  si $\tau$ cambia (e.g., a 0.1 o 6.1). Mire a ver que pasa.


**Tu respuesta:**

1.

2.

---
La siguiente celda calcula la persistencia homologica de la nube de ventanas deslizantes

In [ ]:
## Persistencia homologica de SW_f

# Serie de tiempo
period = 2*np.pi   ## <-- sin(t) is periodic with period 2pi
noise_level = 0.1

t_vals = np.linspace(0 , 5*period, 1000)
y_vals  = np.sin(t_vals) + noise_level*np.random.randn(*t_vals.shape)
f = np.array([t_vals, y_vals]) ## <---- serie de tiempo

# Parametros para SW
d = 2
tau = period/(d+1)  ## <--- la teoria de SW dice que estos son los mejores parametros

# Calcular SW
SW_f = SW_cloud(f,tau, d, n_data )

# Parametros para ripser/ persistencia
max_alpha = 1.8
max_homology_dim = 1

# Correr Ripser
rips_persistence = ripser(SW_f,  maxdim=max_homology_dim , thresh = max_alpha , n_perm =200)

# Visualizar barcodes
dgms = rips_persistence['dgms']
plot_barcodes(dgms,max_alpha);

La longitud de la barra mas larga (su persistencia) esta calculada abajo

In [ ]:
# Persistencia 1-dimensional maxima
max_pers_dim_1 = np.max(dgms[1][:, 1] - dgms[1][:, 0])
print('La persistencia maxima en bcd_1 es', max_pers_dim_1)

**Preguntas:**

1. Si nos concentramos en  $\mathrm{bcd}_1^\mathcal{R}(\mathbb{SW}_{d,\tau}f)$ que le pasa a la longitud de la barra mas larga  (i.e.,  a su <u>persistencia</u>) si  $\tau$ toma valores sub-optimos  (e.g., 6.0 o 0.1). Mire que pasa.

2. Que pasa si el nivel de ruido de la serie de tiempo aumenta? Mire que pasa.

**Tu respuesta:**

1.  

2.

---

# Part III: Sliding Windows and 1-Persistence Scoring (SW1PerS)

Esta actividad recoge todo lo aprendido

Primero generamos varias series de tiempo preiodicas con varios niveles de ruido, el nivel de ruido (menor a mayor) nos da el ranking the periodicidad `true_ranking` de la series de tiempo .

In [ ]:
# Generate a list of synthetic periodic signals corrupted with varying levels of random noise
np.random.seed(2)
t_vals = np.linspace(0,2*np.pi, 1000)

noise = np.random.randn(*t_vals.shape)
noise = noise/np.max(np.abs(noise))

n_signals = 20  # number of signals to be generated

Y_vals = []

for i, noise_level in enumerate(np.linspace(0,1,n_signals)):
  y_vals = (1- noise_level)*np.sin(5*t_vals - np.pi*np.random.rand()) + noise_level*noise
  Y_vals.append(y_vals)

true_ranking = np.random.permutation(n_signals) +1

Y_vals = np.array([Y_vals[i-1] for i in true_ranking])


# Plot the first few signals
n_plots = 7
plt.figure(figsize = (5, 5))

for i in range(n_plots):
  plt.subplot(n_plots, 1, i+1)
  plt.plot(t_vals, Y_vals[i])

In [ ]:
print('El ranking de estas sen~ales de acuerdo a su periodicidad: ', true_ranking[0:n_plots])

## Comparando Rankings:

El *tau de Kendall* es una medida de similitud entre dos rankings:

$$\mathrm{kendall(ranking_1, ranking_2)} \; \in \; [-1, 1]$$

es un numero entre  1 (el mismo ranking) y -1 (rankings inversos).  

In [ ]:
# Compare dos rankings usando el tau de Kendall:

from scipy.stats import kendalltau

random_ranking = np.arange(n_signals).tolist()

k_tau, _ =  kendalltau(true_ranking, random_ranking)

print('La similitud entre el ranking real y el aleatorio es ', k_tau)


**Actividad**

1. Aplique el algoritmo de SW1PerS (nube de ventanas deslizantes --> persistencia homologica --> longitud de la barra 1-dimensional mas larga) para calcular un puntage de periodicidad para cada serie de tiempo en Y_vals. **Stop and think:** usaste los parametros apropiados de $\tau$ y $d$?

2. La lista de puntages de periodicidad calculada en la primera parte de esta actividad se puede usar para derivar un `TDA_ranking` (mayor a menor periodicidad). Cual es la similitud entre este ranking y el original (`true_ranking`)? **Stop and think:** Hay alguna innovacion que puedas aplicar para mejorar la similitud entre estos rankings? *Hint:* Ruido.